# TMDB Movie Data Analysis

This notebook fetches movie data from the TMDB API, cleans it, calculates KPIs and visualises the results.
The reusable code lives in `src/tmdb_analysis/`; this notebook runs it step by step and explains the results.

## Setup

In [1]:
import sys
from pathlib import Path

import pandas as pd

# The notebook runs from notebooks/, so go up one folder to find src/
sys.path.insert(0, str(Path.cwd().parent / "src"))

from tmdb_analysis import config
from tmdb_analysis.api_client import load_movies
from tmdb_analysis.cleaning import (
    drop_irrelevant_columns,
    flatten_json_columns,
    convert_dtypes,
    replace_zeros_with_nan,
    convert_to_musd,
    convert_to_musd,
    fix_zero_votes,
    replace_text_placeholders,

)

pd.set_option("display.max_columns", 30)

## Phase 1: Data Extraction

Movies are fetched from `/movie/{id}?append_to_response=credits`, so details and cast/crew come back in one request.
Each request has a 10-second timeout and is retried up to 3 times on rate limits or server errors.
ID `0` from the brief doesn't exist, so it is skipped. Six extra IDs are added because none of the lab
movies match the two required searches. The raw data is saved to `data/raw/`, so later runs don't call the API again.

In [2]:
df = load_movies()   # use load_movies(refresh=True) to call the API again

print(f"{len(df)} movies, {df.shape[1]} columns")
df.head(2)

Loading saved data from C:\Users\KING\Documents\AmaliTech\Specialization\pandas_numpys\TMDB-Movie-Data-Analysis\data\raw\movies_raw.json
24 movies, 28 columns


,adult,backdrop_path,belongs_to_collection,budget,genres,homepage,id,imdb_id,origin_country,original_language,original_title,overview,popularity,poster_path,production_companies,production_countries,release_date,revenue,runtime,softcore,spoken_languages,status,tagline,title,video,vote_average,vote_count,credits
0,False,/7RyHsO4yDXtBv1zUU3mTpHeQ0d5.jpg,"{'id': 86311, 'name': 'The Avengers Collection...",356000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 878, ...",https://www.marvel.com/movies/avengers-endgame,299534,tt4154796,[US],en,Avengers: Endgame,After the devastating events of Avengers: Infi...,82.3789,/bR8ISy1O9XQxqiy0fQFw2BX72RQ.jpg,"[{'id': 420, 'logo_path': '/hUzeosd33nzE5MCNsZ...","[{'iso_3166_1': 'US', 'name': 'United States o...",2019-04-24,2938567784,181,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,Avenge the fallen.,Avengers: Endgame,False,8.247,29084,"{'cast': [{'adult': False, 'gender': 2, 'id': ..."
1,False,/vL5LR6WdxWPjLPFRLe133jXWsh5.jpg,"{'id': 87096, 'name': 'Avatar Collection', 'po...",237000000,"[{'id': 878, 'name': 'Science Fiction'}, {'id'...",https://www.avatar.com/movies/avatar,19995,tt0499549,[US],en,Avatar,"In the 22nd century, a paraplegic Marine is di...",37.2664,/gKY6q7SjCkAU6FqvqWybDYgUKIF.jpg,"[{'id': 444, 'logo_path': None, 'name': 'Dune ...","[{'iso_3166_1': 'US', 'name': 'United States o...",2009-12-16,2923706026,162,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,Enter the world of Pandora.,Avatar,False,7.611,34892,"{'cast': [{'adult': False, 'gender': 2, 'id': ..."


## Phase 2: Data Cleaning

### 2.1 Drop irrelevant columns and flatten the JSON columns

Nested columns like `genres` become text such as `Action|Adventure`. Empty values become `NaN` (not `""`)
so they count as missing later. The `credits` column is split into `cast`, `cast_size`, `director` and `crew_size`.

In [3]:
df = drop_irrelevant_columns(df)
df = flatten_json_columns(df)

df[["title", "genres", "belongs_to_collection", "cast_size", "director", "crew_size"]].head()

,title,genres,belongs_to_collection,cast_size,director,crew_size
0,Avengers: Endgame,Adventure|Science Fiction|Action,The Avengers Collection,103,Anthony Russo|Joe Russo,613
1,Avatar,Science Fiction|Action|Adventure,Avatar Collection,68,James Cameron,999
2,Star Wars: The Force Awakens,Adventure|Action|Science Fiction,Star Wars Collection,99,J.J. Abrams,270
3,Avengers: Infinity War,Adventure|Action|Science Fiction,The Avengers Collection,69,Anthony Russo|Joe Russo,738
4,Titanic,Drama|Romance,NaN,116,James Cameron,271


### 2.2 Inspect the extracted columns

Multi-value columns are split first, so each genre, language or company is counted on its own.

In [4]:
for column in ["genres", "spoken_languages", "production_countries", "production_companies"]:
    print(df[column].str.split("|").explode().value_counts().head(), "\n")

print(df["belongs_to_collection"].value_counts().head())

genres
Adventure          17
Action             15
Science Fiction    13
Thriller            6
Family              5
Name: count, dtype: int64 

spoken_languages
English     24
Español      4
Français     4
Pусский      4
             3
Name: count, dtype: int64 

production_countries
United States of America    23
United Kingdom               2
France                       1
Name: count, dtype: int64 

production_companies
Marvel Studios              5
Universal Pictures          3
Miramax                     3
A Band Apart                3
Lightstorm Entertainment    2
Name: count, dtype: int64 

belongs_to_collection
The Avengers Collection     4
Star Wars Collection        2
Jurassic Park Collection    2
Frozen Collection           2
Kill Bill Collection        2
Name: count, dtype: int64


## Key Details

Most Appeared Genre: Adventure~17,

Company Dominant ~ 5


### 2.3 Fix data types, replace unrealistic zeros, convert to millions

A budget, revenue or runtime of 0 means the value is unknown, so it becomes `NaN`.
This must happen **before** converting to millions; otherwise an unknown budget becomes `0.0`,
which makes ROI infinite and profit equal to the whole revenue.

In [5]:
print("Zero budgets before cleaning:", (df["budget"] == 0).sum())

df = convert_dtypes(df)
df = replace_zeros_with_nan(df)
df = convert_to_musd(df)

print("Zero values left in budget_musd:", (df["budget_musd"] == 0).sum())
print("Unknown budgets (NaN):", df["budget_musd"].isna().sum())
df[["title", "release_date", "budget_musd", "revenue_musd", "runtime"]].head()

Zero budgets before cleaning: 0
Zero values left in budget_musd: 0
Unknown budgets (NaN): 0


,title,release_date,budget_musd,revenue_musd,runtime
0,Avengers: Endgame,2019-04-24,356.0,2938.567784,181
1,Avatar,2009-12-16,237.0,2923.706026,162
2,Star Wars: The Force Awakens,2015-12-15,245.0,2068.223624,136
3,Avengers: Infinity War,2018-04-25,300.0,2052.415039,149
4,Titanic,1997-12-18,200.0,2264.162353,194


### 2.4 Zero votes and placeholder text

A `vote_average` from zero votes is not a real rating, so it becomes `NaN`; this keeps unrated movies
out of the rating rankings. Placeholder text such as "No Data" in `overview` and `tagline` also becomes `NaN`,
so missing text is counted as missing.

In [6]:
print("Movies with 0 votes:", (df["vote_count"] == 0).sum())

df = fix_zero_votes(df)
df = replace_text_placeholders(df)

print("Missing overview:", df["overview"].isna().sum())
print("Missing tagline:", df["tagline"].isna().sum())

Movies with 0 votes: 0
Missing overview: 0
Missing tagline: 0
